# VascuMap Batch Pipeline

Runs the full VascuMap pipeline on every image in a source folder.  
- **LIF files**: iterates through each image within the file  
- **TIF/TIFF files**: one pipeline run per file  
- Per-image output folder under `Z:\Bel\Vascumap_Outputs\<image_name>\`

**Default outputs** (all 3-D volumes at 2 µm isotropic, aligned for napari overlay):
| File | Description |
|------|-------------|
| `*_overlay_geometry_0.tif` | 2-D device overlay with inner/outer geometry |
| `*_cropped_stack_aligned.npy` | Brightfield stack at 2 µm iso |
| `*_vessel_translation_aligned.npy` | Pix2Pix image translation |
| `*_clean_segmentation.npy` | Cleaned binary vessel segmentation |
| `*_skeleton.npy` | Skeleton from pruned graph |
| `*_analysis_metrics.csv` | Quantitative vascular network metrics |

**Extra outputs** (when `save_all_interim = True`):
| File | Description |
|------|-------------|
| `*_holes.npy` | Binary pore mask |
| `*_hole_labels_per_slice.npy` | Per-slice labelled pores |
| `*_hole_distance_per_slice_um.npy` | Inscribed-radius distance map |
| `*_full_graph_skeleton.npy` | Pre-pruning skeleton |
| `*_vessel_mask.npy` | Raw vessel mask |
| `*_graph_nodes.npz` | Sprout/junction node coords |
| `*_clean_graph.pkl` | NetworkX graph (pickle) |

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Point source_dir at the folder containing your .lif / .tif / .tiff files.
# All outputs land under output_base in per-image sub-folders.

source_dir  = r"Z:\Bel\Vascumap_Example_Lifs\training_data\original_images\test_in\second_run"    
output_base = r"Z:\Bel\Vascumap_Outputs"

# Set True for images whose filename contains "marina" (enables organoid masking).
# If None, the notebook auto-detects from the filename.
force_mask_central_region = None   # True / False / None (auto)

# Device width in micrometres (passed to device segmentation)
device_width_um = 35.0

# Channel index to use for multi-channel images (0-based, default 0)
channel = 0

# Set True to save extra volumes for full napari visualisation
# (holes, pore labels, full-graph skeleton, graph node coords, etc.)
save_all_interim = True

In [2]:
# ── Imports ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add bel_vascumap to the path so we can import the pipeline
sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

import numpy as np
import tifffile
from liffile import LifFile

from vascumap import VascuMap
from utils import resize_dask

W0317 12:49:05.153000 619616 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ── Discover image files and build job list ────────────────────────────────────
source = Path(source_dir)
if not source.is_dir():
    raise FileNotFoundError(f"source_dir does not exist: {source}")

image_files = sorted(
    p for p in source.iterdir()
    if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff")
)

# Each job is (path, image_index, should_mask)
jobs = []
for p in image_files:
    should_mask = (
        force_mask_central_region
        if force_mask_central_region is not None
        else ("marina" in p.name.lower())
    )
    if p.suffix.lower() == ".lif":
        try:
            with LifFile(p) as lif:
                n_images = len(lif.images)
            for idx in range(n_images):
                jobs.append((p, idx, should_mask))
        except Exception as exc:
            print(f"[SKIP] Could not inspect {p.name}: {exc}")
    else:
        jobs.append((p, 0, should_mask))

print(f"Source: {source}")
print(f"Files found: {len(image_files)}  |  Total jobs: {len(jobs)}")
for i, (p, idx, mask) in enumerate(jobs):
    tag = f" (LIF image {idx})" if p.suffix.lower() == ".lif" else ""
    print(f"  {i+1}. {p.name}{tag}  mask_central_region={mask}")

Source: Z:\Bel\Vascumap_Example_Lifs\training_data\original_images\test_in\second_run
Files found: 4  |  Total jobs: 4
  1. farid_new_flow_double_channel.tif  mask_central_region=False
  2. farid_new_flow_double_channel_2.tif  mask_central_region=False
  3. farid_new_flow_single_channel.tif  mask_central_region=False
  4. farid_new_flow_single_channel_2.tif  mask_central_region=False


In [4]:
# ── Processing function ────────────────────────────────────────────────────────

def process_and_save(image_path: Path, image_index: int, output_base: str,
                     mask_central_region: bool = False,
                     save_all_interim: bool = False,
                     channel: int = 0):
    """Run full VascuMap pipeline and save aligned outputs to a per-image folder."""
    vm = VascuMap(
        use_device_segmentation_app=False,
        image_source_path=str(image_path),
        image_index=image_index,
        device_width_um=device_width_um,
        mask_central_region=mask_central_region,
        channel=channel,
    )

    # Build a short human-readable name
    if image_path.suffix.lower() == ".lif":
        vm.image_name = f"{image_path.stem}_img{image_index}_{vm.image_name or 'image'}"
    else:
        vm.image_name = f"{image_path.stem}_{vm.image_name or 'image'}"

    output_dir = Path(output_base) / vm.image_name
    print(f"  Output → {output_dir}")

    vm.pipeline(output_dir=output_dir, save_all_interim=save_all_interim)
    return vm

In [ ]:
# ── Run batch ──────────────────────────────────────────────────────────────────
MAX_RETRIES = 2
results = []          # (short_name, status, message)

for i, (image_path, image_index, mask_flag) in enumerate(jobs, start=1):
    tag = f" (LIF #{image_index})" if image_path.suffix.lower() == ".lif" else ""
    print(f"\n{'='*70}")
    print(f"[{i}/{len(jobs)}] {image_path.name}{tag}  mask_central_region={mask_flag}")
    print(f"{'='*70}")

    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            vm = process_and_save(
                image_path=image_path,
                image_index=image_index,
                output_base=output_base,
                mask_central_region=mask_flag,
                save_all_interim=save_all_interim,
                channel=channel,
            )
            results.append((vm.image_name or image_path.stem, "OK", ""))
            last_exc = None
            break
        except Exception as exc:
            last_exc = exc
            if attempt < MAX_RETRIES:
                print(f"  ⚠ Attempt {attempt} failed: {exc}  — retrying...")
            else:
                print(f"  ✗ FAILED after {MAX_RETRIES} attempts: {exc}")

    if last_exc is not None:
        results.append((image_path.name + tag, "FAILED", str(last_exc)))

# ── Summary ───────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
n_ok = sum(1 for _, s, _ in results if s == "OK")
print(f"Batch complete: {n_ok}/{len(results)} succeeded")
if any(s == "FAILED" for _, s, _ in results):
    print("\nFailed images:")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  - {name}: {msg}")


[1/4] farid_new_flow_double_channel.tif  mask_central_region=False
  Output → Z:\Bel\Vascumap_Outputs\farid_new_flow_double_channel_farid_new_flow_double_channel
Initial z votes {0: 0, 1: 0, 2: 1, 3: 0, 4: 0, 5: 1, 6: 0, 7: 0, 8: 3, 9: 4, 10: 2, 11: 6, 12: 7, 13: 3, 14: 10, 15: 6, 16: 13, 17: 4, 18: 2, 19: 15, 20: 10, 21: 7, 22: 2, 23: 4, 24: 0, 25: 0, 26: 0, 27: 0, 28: 0, 29: 0, 30: 0, 31: 0, 32: 0, 33: 0}
First cropping to z: {7: 0, 8: 3, 9: 4, 10: 2, 11: 6, 12: 7, 13: 3, 14: 10, 15: 6, 16: 13, 17: 4, 18: 2, 19: 15, 20: 10, 21: 7, 22: 2, 23: 4}
Stack width 169.99216969696968


  0%|          | 0/1549 [00:00<?, ?it/s]

## Quick-load aligned outputs in napari

Run the cell below to open a completed image's aligned outputs in napari.  
All volumes share the same shape and 2 µm isotropic voxel size, so they overlay directly.

In [ ]:
# ── View aligned outputs in napari ─────────────────────────────────────────────
import napari
import pickle
import colorsys
import matplotlib.colors as mcolors

# Point at one image's output folder
view_dir = Path(output_base) / "REPLACE_WITH_IMAGE_FOLDER_NAME"  # <-- EDIT THIS

def custom_napari_palette(n_labels, base_colour, hue_span=0.02, sat_span=0.45,
                          light_span=0.45, seed=0, alpha=1.0):
    n_labels = max(1, int(n_labels))
    rng = np.random.default_rng(seed)
    r, g, b = mcolors.to_rgb(base_colour)
    base_h, base_l, base_s = colorsys.rgb_to_hls(r, g, b)
    hue = (base_h + rng.uniform(-hue_span, hue_span, n_labels)) % 1.0
    sat = np.clip(base_s + rng.uniform(-sat_span, sat_span, n_labels), 0.20, 1.00)
    light = np.clip(base_l + rng.uniform(-light_span, light_span, n_labels), 0.18, 0.88)
    order = rng.permutation(n_labels)
    cmap = {0: np.array([0., 0., 0., 0.])}
    for idx, j in enumerate(order, start=1):
        pr, pg, pb = colorsys.hls_to_rgb(float(hue[j]), float(light[j]), float(sat[j]))
        cmap[idx] = np.array([pr, pg, pb, float(alpha)])
    return cmap

# ── Load and display ──────────────────────────────────────────────────────
viewer = napari.Viewer(ndisplay=3)
voxel = (2.0, 2.0, 2.0)

def _load(name):
    p = view_dir / name
    return np.load(str(p)) if p.exists() else None

# Core layers
seg = _load([f for f in view_dir.glob("*_clean_segmentation.npy")][0].name) if list(view_dir.glob("*_clean_segmentation.npy")) else None
skel = _load([f for f in view_dir.glob("*_skeleton.npy")][0].name) if list(view_dir.glob("*_skeleton.npy")) else None
trans = _load([f for f in view_dir.glob("*_vessel_translation_aligned.npy")][0].name) if list(view_dir.glob("*_vessel_translation_aligned.npy")) else None
stack = _load([f for f in view_dir.glob("*_cropped_stack_aligned.npy")][0].name) if list(view_dir.glob("*_cropped_stack_aligned.npy")) else None

if stack is not None:
    viewer.add_image(stack, name='cropped_stack', scale=voxel)
if trans is not None:
    viewer.add_image(trans, name='vessel_translation', scale=voxel)
if seg is not None:
    viewer.add_labels(seg.astype(np.int32), name='clean_segmentation', scale=voxel,
                      colormap=custom_napari_palette(1, 'dodgerblue', 0, 0, 0), opacity=0.35)
if skel is not None:
    viewer.add_labels(skel.astype(np.int32), name='clean_graph_skeleton', scale=voxel,
                      colormap=custom_napari_palette(1, 'darkviolet', 0, 0, 0), opacity=1)

# Interim layers (only present when save_all_interim was True)
holes_f = list(view_dir.glob("*_holes.npy"))
if holes_f:
    from skimage.measure import label as sk_label
    holes = np.load(str(holes_f[0]))
    holes_labels = sk_label(holes)
    viewer.add_labels(holes_labels.astype(np.int32), name='holes', scale=voxel,
                      colormap=custom_napari_palette(len(np.unique(holes_labels)), 'red', 0.1, 0.1, 0.1), opacity=0.6)

hl_f = list(view_dir.glob("*_hole_labels_per_slice.npy"))
if hl_f:
    hl = np.load(str(hl_f[0]))
    viewer.add_labels(hl.astype(np.int32), name='pore_labels', scale=voxel,
                      colormap=custom_napari_palette(int(hl.max()), 'limegreen', 0.2, 0.5, 0.5), opacity=0.8)

hd_f = list(view_dir.glob("*_hole_distance_per_slice_um.npy"))
if hd_f:
    viewer.add_image(np.load(str(hd_f[0])).astype(np.float32), name='hole_distance_um', scale=voxel,
                     colormap='greens', opacity=0.45)

full_skel_f = list(view_dir.glob("*_full_graph_skeleton.npy"))
if full_skel_f:
    viewer.add_labels(np.load(str(full_skel_f[0])), name='full_graph_skeleton', scale=voxel,
                      colormap=custom_napari_palette(1, 'gold', 0, 0, 0), opacity=1)

nodes_f = list(view_dir.glob("*_graph_nodes.npz"))
if nodes_f:
    data = np.load(str(nodes_f[0]))
    pts, is_sprout = data['pts'], data['is_sprout'].astype(bool)
    junction_pts = pts[~is_sprout]
    sprout_pts = pts[is_sprout]
    if len(junction_pts) > 0:
        viewer.add_points(junction_pts, name='junctions', face_color='limegreen', size=5, scale=voxel)
    if len(sprout_pts) > 0:
        viewer.add_points(sprout_pts, name='sprouts', face_color='cyan', size=5, scale=voxel)

napari.run()